In [ ]:
# Parameters
num_qubits = 3
function_type = "Balanced"


In [ ]:
%matplotlib inline
import matplotlib
matplotlib.use('agg')
import warnings
warnings.filterwarnings('ignore')


In [ ]:

from qiskit import QuantumCircuit, transpile
from qiskit_aer import Aer
from qiskit.visualization import circuit_drawer, plot_histogram, plot_bloch_multivector
from qiskit.quantum_info import Statevector, partial_trace
import matplotlib.pyplot as plt
import numpy as np

num_qubits = int(num_qubits)
function_type = str(function_type)
total_qubits = num_qubits + 1

qc = QuantumCircuit(total_qubits, num_qubits)
# Prepare ancilla in |1⟩
qc.x(num_qubits)
qc.barrier()
# Apply H to all
for i in range(total_qubits):
    qc.h(i)
qc.barrier()

# Oracle
if function_type == "Constant":
    pass  # f(x) = 0 for all x
else:
    # Balanced: CNOT from each input qubit to ancilla
    for i in range(num_qubits):
        qc.cx(i, num_qubits)
qc.barrier()

# Apply H to input qubits
for i in range(num_qubits):
    qc.h(i)
# Measure input qubits
for i in range(num_qubits):
    qc.measure(i, i)

print(f"Deutsch-Jozsa on {num_qubits} qubits — Function: {function_type}")

fig = circuit_drawer(qc, output='mpl')
display(fig)
plt.close(fig)

simulator = Aer.get_backend('aer_simulator')
compiled = transpile(qc, simulator)
job = simulator.run(compiled, shots=1000)
result = job.result()
counts = result.get_counts()

all_zeros = '0' * num_qubits
is_constant = counts.get(all_zeros, 0) > 500
print(f"Result: Function is {'CONSTANT' if is_constant else 'BALANCED'}")
print(f"Counts: {counts}")

fig2 = plot_histogram(counts)
display(fig2)
plt.close(fig2)

# Bloch sphere for qubit 0
qc_sv = QuantumCircuit(total_qubits)
qc_sv.x(num_qubits)
for i in range(total_qubits):
    qc_sv.h(i)
if function_type != "Constant":
    for i in range(num_qubits):
        qc_sv.cx(i, num_qubits)
for i in range(num_qubits):
    qc_sv.h(i)
sv = Statevector.from_instruction(qc_sv)
rho = partial_trace(sv, list(range(1, total_qubits)))
a = np.sqrt(np.real(rho.data[0, 0]))
b_complex = rho.data[1, 0] / a if a > 1e-6 else 0
theta = 2 * np.arccos(np.clip(a, 0, 1))
phi = np.angle(b_complex) % (2 * np.pi)

try:
    fig3 = plot_bloch_multivector(sv)
    display(fig3)
    plt.close(fig3)
except Exception:
    pass
print(f"BLOCH_THETA={theta:.6f}")
print(f"BLOCH_PHI={phi:.6f}")
